# Bullying Detection Challenge

## Your task

Each video clip shows either **bullying** or **no bullying**. Your job is
to build a model that watches a video and predicts which one it is.

- `train/` is labeled: `train/bully/` and `train/no-bully/` contain example
  clips for each class. Use this to train your model.
- `val/` is **unlabeled**: a flat folder of clips with generic names
  (`val_0000.avi`, `val_0001.avi`, ...). You don't get the answers for
  these. Your model predicts a label for every clip in `val/`, and you
  save those predictions to `submission.csv`.
- `submission.csv` must have exactly two columns: `video_id,predicted_label`,
  where `predicted_label` is either `bully` or `no-bully`.

## What's already done for you

Everything through loading the data, turning videos into frames, and
building the DataLoaders is finished and working (Sections 1-7) — you
shouldn't need to touch it. A **sample model** is also included
(Section 8-9) so you have a full working pipeline from day one, all the
way to a real `submission.csv`. It is deliberately simple and scores
poorly on purpose.

**Your job starts at Section 8.** Replace the sample model with your own
approach, or improve on it. Section 10 has some directions to research —
model names only, no hand-holding, that part is on you.

## Section 1 - Install Libraries

In [ ]:
!pip install -q torch torchvision
!pip install -q transformers
!pip install -q opencv-python-headless
!pip install -q scikit-learn
!pip install -q pandas
!pip install -q tqdm


## Section 2 - Import Libraries

In [1]:
import os
import csv
import glob
import random

import numpy as np
import pandas as pd
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from tqdm.auto import tqdm

print("All libraries imported successfully.")


2026-07-12 15:36:15.851623: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-12 15:36:15.859831: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783859775.870181   10968 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783859775.873211   10968 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783859775.881450   10968 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

All libraries imported successfully.


## Section 3 - Configuration

In [12]:
# ----------------------------
# General settings
# ----------------------------
SEED = 42               # random seed, for reproducibility
IMAGE_SIZE = 32          # frames are resized to IMAGE_SIZE x IMAGE_SIZE
NUM_FRAMES = 16            # number of frames sampled from each video
BATCH_SIZE = 4             # videos per training batch

# ----------------------------
# Label mapping
# ----------------------------
LABEL2ID = {"no-bully": 0, "bully": 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# ----------------------------
# Reproducibility
# ----------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------
# GPU check
# ----------------------------
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("GPU detected. Using device:", torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device("cpu")
    print("No GPU detected. Using device: CPU (training will be slower).")


GPU detected. Using device: NVIDIA GeForce RTX 3060


## Section 4 - Locate the Dataset

Auto-detects the dataset under `/kaggle/input/` by looking for a folder
that contains `train/bully` and `train/no-bully`. If it can't find one
(for example if you're running this outside Kaggle), set `DATASET_ROOT`
by hand.

In [13]:
def _find_dataset_root(start_path="/kaggle/input"):
    for root, dirs, files in os.walk(start_path):
        if "train" in dirs and "val" in dirs:
            train_dir = os.path.join(root, "train")
            if "bully" in os.listdir(train_dir) and "no-bully" in os.listdir(train_dir):
                return root
    return None

DATASET_ROOT = _find_dataset_root()

if DATASET_ROOT is None:
    # Couldn't auto-detect it -- set this to your dataset's path by hand.
    DATASET_ROOT = "/kaggle/input/your-dataset-slug-here"

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR = os.path.join(DATASET_ROOT, "val")

print("Dataset root:", DATASET_ROOT)
print("Train dir (labeled):    ", TRAIN_DIR)
print("Val dir (unlabeled):    ", VAL_DIR)


Dataset root: RWF-2000/
Train dir (labeled):     RWF-2000/train
Val dir (unlabeled):     RWF-2000/val


## Section 5 - Load Data (done for you)

In [14]:
def list_labeled_videos(root_dir):
    """(video_path, label) pairs from a bully/no-bully organized folder."""
    items = []
    for class_name, label in LABEL2ID.items():
        class_dir = os.path.join(root_dir, class_name)
        paths = glob.glob(os.path.join(class_dir, "*.avi")) + \
                glob.glob(os.path.join(class_dir, "*.mp4"))
        for p in paths:
            items.append((p, label))
    return items

all_train_items = list_labeled_videos(TRAIN_DIR)
random.shuffle(all_train_items)

# A small slice of the labeled data, held out from training, so you have
# some way to check your own progress. The real val/ folder has no labels
# on purpose -- that's what gets predicted for submission.csv.
LOCAL_VAL_FRACTION = 0.15
split_index = int(len(all_train_items) * (1 - LOCAL_VAL_FRACTION))
local_train_items = all_train_items[:split_index]
local_val_items = all_train_items[split_index:]

submission_video_paths = sorted(
    glob.glob(os.path.join(VAL_DIR, "*.avi")) + glob.glob(os.path.join(VAL_DIR, "*.mp4"))
)

print("Training videos:                 ", len(local_train_items))
print("Local validation videos (yours): ", len(local_val_items))
print("Videos to predict for submission:", len(submission_video_paths))


Training videos:                  1360
Local validation videos (yours):  240
Videos to predict for submission: 400


## Section 6 - Video Preprocessing (done for you)

In [15]:
def extract_frames(video_path, num_frames=NUM_FRAMES, image_size=IMAGE_SIZE):
    """Open a video file and return a list of `num_frames` RGB frames,
    evenly spaced across the video."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        total_frames = num_frames

    frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    frames = []
    current_index = 0
    next_wanted = 0

    while cap.isOpened() and next_wanted < len(frame_indices):
        success, frame = cap.read()
        if not success:
            break
        if current_index == frame_indices[next_wanted]:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (image_size, image_size))
            frames.append(frame)
            next_wanted += 1
        current_index += 1

    cap.release()

    while len(frames) < num_frames:
        frames.append(frames[-1] if frames else np.zeros((image_size, image_size, 3), dtype=np.uint8))

    return frames

MODEL_CHECKPOINT = "MCG-NJU/videomae-base-finetuned-kinetics"
image_processor = VideoMAEImageProcessor.from_pretrained(MODEL_CHECKPOINT)

def process_frames(frames):
    """Turn a list of RGB frames into a (num_frames, C, H, W) tensor."""
    processed = image_processor(list(frames), return_tensors="pt")
    return processed["pixel_values"][0]


## Section 7 - Datasets & DataLoaders (done for you)

In [16]:
class LabeledVideoDataset(Dataset):
    """For training / your own local validation: returns (video_tensor, label)."""
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        video_path, label = self.items[index]
        frames = extract_frames(video_path)
        video_tensor = process_frames(frames)
        return video_tensor, label

class UnlabeledVideoDataset(Dataset):
    """For the videos you must submit predictions for: returns
    (video_tensor, video_id). There is no label here on purpose."""
    def __init__(self, video_paths):
        self.video_paths = video_paths

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, index):
        video_path = self.video_paths[index]
        video_id = os.path.basename(video_path)
        frames = extract_frames(video_path)
        video_tensor = process_frames(frames)
        return video_tensor, video_id

train_dataset = LabeledVideoDataset(local_train_items)
local_val_dataset = LabeledVideoDataset(local_val_items)
submission_dataset = UnlabeledVideoDataset(submission_video_paths)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
local_val_loader = DataLoader(local_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
submission_loader = DataLoader(submission_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Training batches:  ", len(train_loader))
print("Local val batches: ", len(local_val_loader))
print("Submission batches:", len(submission_loader))


Training batches:   340
Local val batches:  60
Submission batches: 100


## Section 8 - Sample Baseline Model (⚠️ your work starts here)

This is a real model (VideoMAE), but almost all of it is **frozen** —
only the final classification layer trains, for a single epoch. It exists
so you have a complete, working pipeline (train → predict → `submission.csv`)
from the start. It will score only a little better than guessing.

Replace or rebuild this section with your own model.

In [17]:
model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

# --- Intentionally weak: only the classifier head is trainable ---
for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

BASELINE_EPOCHS = 1
LEARNING_RATE = 1e-6

model.to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Baseline model loaded. Trainable parameters: {trainable:,} / {total:,}")


Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base-finetuned-kinetics and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([400]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([400, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Baseline model loaded. Trainable parameters: 1,538 / 86,228,738


## Section 9 - Train the Baseline

In [18]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE
)

def run_training_epoch(loader):
    model.train()
    total_loss = 0.0
    for video_tensors, labels in tqdm(loader, desc="Training"):
        video_tensors = video_tensors.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(pixel_values=video_tensors)
        loss = loss_function(outputs.logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(loader):
    """Accuracy + F1 on a LabeledVideoDataset loader."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for video_tensors, labels in tqdm(loader, desc="Evaluating"):
            video_tensors = video_tensors.to(DEVICE)
            outputs = model(pixel_values=video_tensors)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    _, _, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )
    return acc, f1

for epoch in range(1, BASELINE_EPOCHS + 1):
    train_loss = run_training_epoch(train_loader)
    local_acc, local_f1 = evaluate(local_val_loader)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}  "
          f"local_val_acc={local_acc:.4f}  local_val_f1={local_f1:.4f}")


Training:   0%|          | 0/340 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 1: train_loss=0.7077  local_val_acc=0.4917  local_val_f1=0.5643


## Section 10 - Ideas to Research

The baseline above is intentionally weak. Some directions worth
researching and trying on your own (find the details, docs, and code
yourself):

- Unfreeze more layers of VideoMAE, or fine-tune the whole backbone
  instead of just the classifier head.
- Train for more epochs, and try a learning-rate schedule.
- Other video architectures: **3D CNNs (I3D, R3D, C3D)**, **X3D**,
  **SlowFast**, **TimeSformer**, **ViViT**.
- A simpler route: a 2D CNN per frame combined with an **LSTM or GRU**
  over time.
- Data augmentation: random crops, flips, temporal jitter, sampling a
  different number of frames.
- Class balance: check whether `bully` and `no-bully` are evenly
  represented in your training data, and weight the loss if not.
- Ensembling multiple models or multiple frame samples per video.

Use `local_val_loader` (Section 9) to check your own progress as you
iterate — you won't see the real `val/` labels, so this is your only
feedback loop until you produce a new `submission.csv`.

## Section 11 - Generate submission.csv

In [19]:
def predict_for_submission(loader):
    model.eval()
    video_ids, predicted_labels = [], []
    with torch.no_grad():
        for video_tensors, ids in tqdm(loader, desc="Predicting"):
            video_tensors = video_tensors.to(DEVICE)
            outputs = model(pixel_values=video_tensors)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            video_ids.extend(ids)
            predicted_labels.extend(ID2LABEL[p] for p in preds)
    return video_ids, predicted_labels

video_ids, predicted_labels = predict_for_submission(submission_loader)

SUBMISSION_PATH = "submission.csv"
with open(SUBMISSION_PATH, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["video_id", "predicted_label"])
    writer.writerows(zip(video_ids, predicted_labels))

print("Saved", SUBMISSION_PATH, "with", len(video_ids), "predictions.")
pd.read_csv(SUBMISSION_PATH).head()


Predicting:   0%|          | 0/100 [00:00<?, ?it/s]

Saved submission.csv with 400 predictions.


,video_id,predicted_label
0,val_0000.avi,bully
1,val_0001.avi,no-bully
2,val_0002.avi,no-bully
3,val_0003.avi,bully
4,val_0004.avi,bully


---
### Before you submit

- `submission.csv` needs one row per video in `val/`, with columns
  `video_id,predicted_label`.
- `video_id` must exactly match the filenames in `val/` (e.g. `val_0000.avi`).
- `predicted_label` must be exactly `bully` or `no-bully`.